# Data Preprocessing and Feature Extraction
In this notebook, we load the sensor data, window it, extract time and frequency features, and normalize the dataset.

In [1]:
import os
import glob
import pandas as pd
import numpy as np
from scipy.fftpack import fft
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

data_dir = './data'
activities = ['standing', 'walking', 'jumping', 'still']

def compute_features(window_df):
    features = {}
    axes = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
    for axis in axes:
        if axis not in window_df.columns:
            continue
        data = window_df[axis].values
        if len(data) == 0:
            continue
            
        # Time-domain features
        features[f'{axis}_mean'] = np.mean(data)
        features[f'{axis}_var'] = np.var(data)
        features[f'{axis}_std'] = np.std(data)
        features[f'{axis}_rms'] = np.sqrt(np.mean(data**2))
        
        # Frequency-domain features
        fft_vals = np.abs(fft(data))
        freqs = np.fft.fftfreq(len(data), d=1/50.0) # 50 Hz sampling rate
        
        pos_mask = freqs > 0
        if np.any(pos_mask):
            pos_fft = fft_vals[pos_mask]
            pos_freqs = freqs[pos_mask]
            features[f'{axis}_dom_freq'] = pos_freqs[np.argmax(pos_fft)]
            features[f'{axis}_spectral_energy'] = np.sum(pos_fft**2) / len(pos_fft)
        else:
            features[f'{axis}_dom_freq'] = 0
            features[f'{axis}_spectral_energy'] = 0
            
    return features

all_features = []

for activity in activities:
    activity_dir = os.path.join(data_dir, activity)
    if not os.path.exists(activity_dir):
        continue
        
    sessions = [d for d in os.listdir(activity_dir) if os.path.isdir(os.path.join(activity_dir, d))]
    
    for session in sessions:
        session_dir = os.path.join(activity_dir, session)
        acc_path = os.path.join(session_dir, 'Accelerometer.csv')
        gyro_path = os.path.join(session_dir, 'Gyroscope.csv')
        
        if not (os.path.exists(acc_path) and os.path.exists(gyro_path)):
            continue
            
        acc_df = pd.read_csv(acc_path)
        gyro_df = pd.read_csv(gyro_path)
        
        acc_df = acc_df.rename(columns={'x': 'acc_x', 'y': 'acc_y', 'z': 'acc_z'})
        gyro_df = gyro_df.rename(columns={'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'})
        
        acc_df = acc_df.sort_values('seconds_elapsed')
        gyro_df = gyro_df.sort_values('seconds_elapsed')
        
        df = pd.merge_asof(acc_df[['seconds_elapsed', 'acc_x', 'acc_y', 'acc_z']], 
                           gyro_df[['seconds_elapsed', 'gyro_x', 'gyro_y', 'gyro_z']], 
                           on='seconds_elapsed', direction='nearest')
                           
        df['window_id'] = df['seconds_elapsed'] // 1.0
        
        for window_id, window_data in df.groupby('window_id'):
            if len(window_data) < 10:
                continue
                
            features = compute_features(window_data)
            features['activity'] = activity
            features['session'] = session
            features['window_id'] = window_id
            all_features.append(features)

features_df = pd.DataFrame(all_features)
print(f"Extracted {len(features_df)} windows in total.")
print(features_df['activity'].value_counts())

Extracted 501 windows in total.
activity
still       145
jumping     143
walking     141
standing     72
Name: count, dtype: int64


## Normalization
We apply Z-score normalization to all feature columns.

In [2]:
feature_cols = [c for c in features_df.columns if c not in ['activity', 'session', 'window_id']]

scaler = StandardScaler()
features_df[feature_cols] = scaler.fit_transform(features_df[feature_cols])

features_df.to_csv('processed_features.csv', index=False)
features_df.head()

,acc_x_mean,acc_x_var,acc_x_std,acc_x_rms,acc_x_dom_freq,acc_x_spectral_energy,acc_y_mean,acc_y_var,acc_y_std,acc_y_rms,...,gyro_y_spectral_energy,gyro_z_mean,gyro_z_var,gyro_z_std,gyro_z_rms,gyro_z_dom_freq,gyro_z_spectral_energy,activity,session,window_id
0,0.199155,-0.364417,-0.615847,-0.621081,-0.425662,-0.360108,0.233394,-0.493630,-0.629065,-0.634392,...,-0.469556,0.006358,-0.495382,-0.795488,-0.780648,-0.674009,-0.489064,standing,2026-07-03_12-01-28,0.0
1,0.181841,-0.364427,-0.616525,-0.621461,-0.700650,-0.360102,0.219219,-0.493626,-0.628305,-0.633545,...,-0.469424,0.078531,-0.495346,-0.794060,-0.774960,-0.700655,-0.488983,standing,2026-07-03_12-01-28,1.0
2,0.201249,-0.364335,-0.610782,-0.616086,-0.125634,-0.360000,0.234912,-0.493614,-0.626509,-0.631840,...,-0.469182,0.005297,-0.495113,-0.785854,-0.773488,-0.709851,-0.488724,standing,2026-07-03_12-01-28,2.0
3,0.178809,-0.364446,-0.617904,-0.622707,-0.708853,-0.360114,0.231231,-0.493618,-0.627125,-0.632484,...,-0.469694,0.028109,-0.495158,-0.787347,-0.780143,-0.475547,-0.488771,standing,2026-07-03_12-01-28,3.0
4,0.232387,-0.363953,-0.593437,-0.598596,-0.708853,-0.359611,0.225138,-0.493592,-0.623879,-0.629249,...,-0.469193,0.040780,-0.494742,-0.774931,-0.770465,-0.245750,-0.488349,standing,2026-07-03_12-01-28,4.0
